In [62]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [63]:
pasta_dados = r"C:\Users\Joao.Maria\Documents\curso-python\AnaliseDados\Projeto 1 - Auditoria de Dados Campanha\Dados"

dados_campanha = os.path.join(pasta_dados, "Campanha Distribuidores Vinho.xlsx")
dados_configuracao_campanha = os.path.join(pasta_dados, "Configuração Campanha Distribuidores Vinho.xlsx")
dados_bruto = os.path.join(pasta_dados, "Dados Brutos Fevereiro.csv")

In [64]:
relatorio_campanha_premiacao = pd.read_excel(dados_campanha, sheet_name="Premiação Distribuidores")
relatorio_campanha_detalhe = pd.read_excel(dados_campanha, sheet_name="Detalhamento Mês Apurado")
configuracao_campanha_cliente = pd.read_excel(dados_configuracao_campanha, sheet_name="Cliente")
configuracao_campanha_produto = pd.read_excel(dados_configuracao_campanha, sheet_name="Produto")
relatorio_bruto = pd.read_csv(dados_bruto, sep=";")

In [65]:
def contagem_validacao(validacao):

    """
    Conta a quantidade de resultados "Correto" e "Divergente" em uma validação.
    Deve conter o campo "Validação" no DataFrame.

    Args:
        validacao (pd.DataFrame): DataFrame geral de validação

    Returns:
        tuple: Uma tupla com a quantidade de itens "Correto" e "Divergente" (int, int).
    """

    try:
        # conta quantas vezes cada valor aparece na coluna
        contagem_validacao = validacao["Validação"].value_counts()

        # valores considerados "corretos"
        valores_corretos = ["Correto", "Ok", "OK", "TRUE"]

        # soma de todos os valores corretos encontrados
        qtde_correta = contagem_validacao.loc[
            contagem_validacao.index.isin(valores_corretos)
        ].sum()

        # divergente = total - corretos
        qtde_divergente = len(validacao) - qtde_correta

        return qtde_correta, qtde_divergente

    except Exception as erro:
        raise Exception(f"Algo de errado aconteceu ao contar valores: {erro}")



def observacao_validacao(validacao, qtde_correto, qtde_divergente):

    """
    Adiciona uma observação ao DataFrame de validação com as quantidades de itens "Correta" e "Divergente".

    Args:
        validacao (pd.DataFrame): DataFrame de validação dos items.
        qtde_correto (int): Quantidade de itens "Correto".
        qtde_divergente (int): Quantidade de itens "Divergente".

    Returns:
        pd.DataFrame: O DataFrame de validação com a coluna "Observação" preenchida.
    """
    
    try:
        validacao.loc[0, "Observação"] = f"Existem {qtde_correto} Correto, e {qtde_divergente} Divergentes"

        return validacao
    
    except Exception as erro:
        raise Exception(f"Ocorreu um erro na hora da adição do campo Observacao ao Dataframe: {erro}")
    


def validar_multiplas_regras(dataframe, regras, separador= " | "):

    """
    Aplica múltiplas regras vetorizadas e retorna uma única coluna:
    - "Correto" se nenhuma regra falhar
    - texto com os motivos se houver divergências

    Args:
        dataframe (pd.DataFrame): DataFrame geral de validação
        regras (list -> tuple): Lista com tuplas dentro com validações
        separador (str): Caracter que separa as divergências
    """

    mensagens = np.full(len(dataframe), "", dtype=object)

    for mascara, texto in regras:
        mensagens = np.where(
            mascara,
            np.where(
                mensagens == "",
                texto,
                mensagens + separador + texto
            ),
            mensagens
        )

    return np.where(mensagens == "", "Correto", mensagens)


In [66]:
dados_bruto = pd.merge(
    relatorio_bruto,
    configuracao_campanha_cliente[["CODIGOCLIENTE"]],
    how="inner",
    on="CODIGOCLIENTE"
)

dados_bruto = pd.merge(
    dados_bruto,
    configuracao_campanha_produto[["CODIGOPRODUTO"]],
    how="inner",
    on="CODIGOPRODUTO"
)

dados_bruto = dados_bruto[dados_bruto["CFOP"].isin([5102, 5106, 5110, 6102, 6106, 6110, 5160, 6160, 6910, 5910])]

dados_bruto = dados_bruto.rename(columns={
    "VOLUME": "VOLUME BRUTO"
})

### 1. Validação Base Apurada x Base Bruta

In [67]:
validacao_volume_campanha_bruto = pd.merge(
    relatorio_campanha_detalhe,
    dados_bruto[["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO", "VOLUME BRUTO"]],
    how="inner",
    on=["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO"]
)

validacao_volume_campanha_bruto["Validação"] = np.where(
    validacao_volume_campanha_bruto["VOLUME"] != validacao_volume_campanha_bruto["VOLUME BRUTO"],
    "Divergente",
    "Correto"
)

qtde_correto_01, qtde_divergente_01 = contagem_validacao(validacao_volume_campanha_bruto)
validacao_volume_campanha_bruto = observacao_validacao(validacao_volume_campanha_bruto, qtde_correto_01, qtde_divergente_01)

### 2. Validação Premiação

In [68]:
PERCENTUAL_CRESCIMENTO_PREMIACAO = [
    {"min": 0.20, "max": float("inf"), "percentual": 0.10},
    {"min": 0.15, "max": 0.1999, "percentual": 0.08},
    {"min": 0.10, "max": 0.1499, "percentual": 0.06},
    {"min": 0.05, "max": 0.0999, "percentual": 0.03},
    {"min": 0.00, "max": 0.0499, "percentual": 0.00},
]

def percentual_premiacao_crescimento(crescimento):
    
    for faixa in PERCENTUAL_CRESCIMENTO_PREMIACAO:
        if faixa["min"] <= crescimento <= faixa["max"]:
            return faixa["percentual"]
        
    return 0.0

In [69]:
campanha_detalhe = (
    relatorio_campanha_detalhe.groupby(["CODIGOCLIENTE", "CODIGOPRODUTO"])
    ["VOLUME"].sum()
    .reset_index()
)

validacao_preamiacao = pd.merge(
    relatorio_campanha_premiacao,
    campanha_detalhe,
    how="left",
    on=["CODIGOCLIENTE", "CODIGOPRODUTO"]
)

validacao_preamiacao["% Crescimento Calculado"] = (
    validacao_preamiacao["VOLUMEREALIZADO"] / validacao_preamiacao["METAVOLUMEVENDA"] - 1
)

validacao_preamiacao["% Premiação Calculado"] = validacao_preamiacao["% Crescimento Calculado"].apply(percentual_premiacao_crescimento)

validacao_preamiacao["Valor Premiacao Calculado"] = (
    validacao_preamiacao["VALORVENDAPREMIACAO"] * validacao_preamiacao["% Premiação Calculado"]
)

validacao_preamiacao["Validação"] = validar_multiplas_regras(
    validacao_preamiacao,
    regras=[
        (
            validacao_preamiacao["% Premiação Calculado"] != validacao_preamiacao["PORCENTAGEMPREMIACAO"], 
            "Percentual Premiação divergente do Cálculo"
        ),
        (
            validacao_preamiacao["VALORPREMIACAO"] != validacao_preamiacao["Valor Premiacao Calculado"], 
            "Valor Premiação divergente do Cálculo"
        ),
    ]
)

qtde_correto_02, qtde_divergente_02 = contagem_validacao(validacao_preamiacao)
validacao_preamiacao = observacao_validacao(validacao_preamiacao, qtde_correto_02, qtde_divergente_02)

In [70]:
with pd.ExcelWriter(r"C:\Users\Joao.Maria\Documents\curso-python\AnaliseDados\Projeto 1 - Auditoria de Dados Campanha\Dados\Indicadores.xlsx", engine="xlsxwriter") as writer:

    validacao_volume_campanha_bruto.to_excel(
        writer,
        sheet_name="Dados Apuração x Bruto",
        index=False
    )
    
    validacao_preamiacao.to_excel(
        writer,
        sheet_name="Premiação",
        index=False
    )